# Semantic Text Similarity & Cosine Similarity
### Sprint 2, Item 1

Sprint 1 already used cosine similarity to rank resumes against job descriptions.
This notebook digs into *why* cosine similarity is the right choice for embeddings,
and compares it against two alternatives: Euclidean distance and raw dot product.

Assumes `embeddings.ipynb` (Sprint 1) has already been run, so
`data/processed/embeddings/resume_embeddings.npy` and `job_embeddings.npy` exist.


In [1]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. Load saved embeddings from Sprint 1

In [2]:
import numpy as np
import pandas as pd

EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "embeddings"

resume_embeddings = np.load(EMBEDDINGS_DIR / "resume_embeddings.npy")
job_embeddings = np.load(EMBEDDINGS_DIR / "job_embeddings.npy")
resume_index = pd.read_csv(EMBEDDINGS_DIR / "resume_index.csv")
job_index = pd.read_csv(EMBEDDINGS_DIR / "job_index.csv")

print("Resume embeddings:", resume_embeddings.shape)
print("Job embeddings:", job_embeddings.shape)

Resume embeddings: (2481, 384)
Job embeddings: (24, 384)


## 2. Three ways to compare two vectors

- **Cosine similarity**: measures the *angle* between two vectors, ignoring their
  magnitude. Range: -1 to 1 (1 = identical direction).
- **Euclidean distance**: measures the *straight-line* distance between two points.
  Sensitive to vector length/magnitude.
- **Dot product**: sum of element-wise products. Combines both angle and magnitude —
  only equivalent to cosine similarity when vectors are normalized to unit length.

For text embeddings, magnitude often reflects things like sentence length or word
frequency rather than meaning, so cosine similarity (which cancels magnitude out) is
the standard choice — this is why `embeddings.ipynb` normalized vectors before scoring.


In [3]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def euclidean_distance(a, b):
    return norm(a - b)

def dot_product(a, b):
    return np.dot(a, b)

# Compare one resume against one job description all three ways
i, j = 0, 0
r_vec, j_vec = resume_embeddings[i], job_embeddings[j]

print(f"Resume {i} vs Job description {j} ({job_index.iloc[j, 0]}):")
print(f"  Cosine similarity: {cosine_similarity(r_vec, j_vec):.4f}")
print(f"  Euclidean distance: {euclidean_distance(r_vec, j_vec):.4f}  (lower = more similar)")
print(f"  Dot product:        {dot_product(r_vec, j_vec):.4f}")

Resume 0 vs Job description 0 (ACCOUNTANT):
  Cosine similarity: 0.3163
  Euclidean distance: 1.1694  (lower = more similar)
  Dot product:        0.3163


## 3. Demonstrate why magnitude matters
Scaling a vector changes its dot product and Euclidean distance to itself's "twin",
but cosine similarity is unaffected — this is the property that makes it robust for
comparing embeddings of different lengths of text.

In [4]:
scaled_vec = r_vec * 3.0  # same direction, different magnitude

print("Original vs itself scaled 3x:")
print(f"  Cosine similarity: {cosine_similarity(r_vec, scaled_vec):.4f}  (unchanged — same direction)")
print(f"  Euclidean distance: {euclidean_distance(r_vec, scaled_vec):.4f}  (changed — different magnitude)")
print(f"  Dot product:        {dot_product(r_vec, scaled_vec):.4f}  (changed — scales with magnitude)")

Original vs itself scaled 3x:
  Cosine similarity: 1.0000  (unchanged — same direction)
  Euclidean distance: 2.0000  (changed — different magnitude)
  Dot product:        3.0000  (changed — scales with magnitude)


## 4. Rank the same job description three ways
Compares the top-5 resume ranking under each metric to see how much they agree/disagree.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine, euclidean_distances

job_vec = job_embeddings[0]

cos_scores = resume_embeddings @ job_vec  # embeddings are pre-normalized, so dot == cosine
euc_scores = -euclidean_distances(resume_embeddings, job_vec.reshape(1, -1)).flatten()  # negate: higher = closer
dot_scores = resume_embeddings @ job_vec  # same as cosine here since normalized

comparison = resume_index.copy()
comparison["cosine_rank"] = pd.Series(cos_scores).rank(ascending=False)
comparison["euclidean_rank"] = pd.Series(euc_scores).rank(ascending=False)

print(f"Top 5 by cosine similarity for job: {job_index.iloc[0, 0]}")
display(comparison.sort_values("cosine_rank").head(5))

Top 5 by cosine similarity for job: ACCOUNTANT


,ID,Category,cosine_rank,euclidean_rank
1845,20253563,ACCOUNTANT,1.0,1.0
1866,49204385,ACCOUNTANT,2.0,2.0
1822,21794875,ACCOUNTANT,3.0,3.0
1872,29821051,ACCOUNTANT,4.0,4.0
1914,29050809,ACCOUNTANT,5.0,5.0


---
## Takeaway
Because Sprint 1's embeddings were already normalized (`normalize_embeddings=True`),
cosine similarity and dot product give identical rankings here. Euclidean distance
correlates closely too, since normalized vectors constrain the geometry — but cosine
similarity remains the standard choice because it stays correct even if normalization
is ever removed upstream.
